# 08 — Holdout de clase

El día 24 el profesor entrega diez preguntas que nadie ha visto. Este notebook las ejecuta y deja escrita la lectura que entra en la presentación: el acierto, el delta contra el golden propio y qué falló.

No toca el agente. Solo llama a `evaluar()`.

1. Deja el `holdout.jsonl` en esta carpeta (la del proyecto) o en Descargas.
2. **Run All**.
3. Lee en voz alta la última celda. El mismo texto queda en `resultados/holdout_clase_defensa.txt`.

Para ensayarlo esta noche, pon en la celda siguiente `RUTA_HOLDOUT` apuntando a `golden/holdout_simulado.jsonl`. Mañana vuelve a dejarlo en `None` y suelta el fichero del profesor en la carpeta del proyecto: con la ruta puesta, el notebook no busca el fichero nuevo.

Si ese mismo fichero ya se ejecutó, no vuelve a llamar al modelo. Para repetirlo, pon `REEJECUTAR = True`.

In [ ]:
# Únicas dos cosas que hace falta tocar, y solo si el fichero no aparece solo.
RUTA_HOLDOUT = None          # por ejemplo: r"C:\Users\jdmar\Downloads\holdout.jsonl"
REEJECUTAR = False           # True vuelve a pagar las diez preguntas

import hashlib
import json
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def raiz_del_repo() -> Path:
    aqui = Path.cwd().resolve()
    for candidato in [aqui, *aqui.parents]:
        if (candidato / "agente" / "__init__.py").is_file():
            return candidato
    raise FileNotFoundError(
        "No encuentro la carpeta del proyecto. Abre el notebook desde "
        "«Taller NLP» y vuelve a ejecutar."
    )

RAIZ = raiz_del_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from agente import config, interfaz

pd.set_option("display.max_colwidth", 88)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

DESTINO = config.DIR_RESULTADOS / "holdout_clase"
RUTA_META = config.DIR_RESULTADOS / "holdout_clase_meta.json"
RUTA_RECIBIDO = config.DIR_RESULTADOS / "holdout_recibido.jsonl"
RUTA_DEFENSA = config.DIR_RESULTADOS / "holdout_clase_defensa.txt"

# El ensayo del notebook 07 no es el holdout del profesor. No se usa nunca
# como sustituto: presentarlo como si lo fuera mediría el conjunto que
# escribimos nosotros.
EXCLUIR = {
    "golden_set.jsonl",
    "golden_set_propio.jsonl",
    "golden_set_ausencias.jsonl",
    "golden_set_ejemplo.jsonl",
    "holdout_simulado.jsonl",
    "mini_holdout.jsonl",
}


def buscar_holdout() -> Path:
    if RUTA_HOLDOUT:
        ruta = Path(RUTA_HOLDOUT).expanduser()
        if not ruta.is_file():
            raise FileNotFoundError(f"No existe {ruta}")
        return ruta.resolve()

    carpetas = [
        RAIZ,
        Path.home() / "Downloads",
        Path.home() / "Desktop",
        Path.home() / "OneDrive" / "Desktop",
        Path.home() / "OneDrive" / "Downloads",
    ]
    exactos = []
    parecidos = []
    for carpeta in carpetas:
        if not carpeta.is_dir():
            continue
        for fichero in carpeta.glob("*.jsonl"):
            nombre = fichero.name.lower()
            if nombre in EXCLUIR or "simulado" in nombre:
                continue
            fichero = fichero.resolve()
            if nombre in {"holdout.jsonl", "preguntas_ciegas.jsonl"}:
                exactos.append(fichero)
            elif "holdout" in nombre or "cieg" in nombre:
                parecidos.append(fichero)

    # La carpeta del proyecto gana a Descargas: es donde se le dice al
    # profesor que se deja el fichero, y un holdout.jsonl viejo en Descargas
    # no debe pisarlo.
    exactos.sort(key=lambda p: (p.parent != RAIZ, -p.stat().st_mtime))
    if exactos:
        return exactos[0].resolve()
    parecidos.sort(key=lambda p: -p.stat().st_mtime)
    if parecidos:
        return parecidos[0].resolve()
    raise FileNotFoundError(
        "No encuentro el holdout del profesor.\n"
        "Déjalo como holdout.jsonl en la carpeta del proyecto o en Descargas,\n"
        "o escribe la ruta en RUTA_HOLDOUT, al principio de esta celda."
    )


def normalizar(item: dict, numero: int) -> dict:
    """Deja el ítem en el esquema que leen los evaluadores.

    El enunciado fija los nombres. Un fichero del profesor puede llegar con
    una familia acentuada o con el ejercicio como «FY2025», y eso no debe
    tumbar la evaluación. No se inventa ningún campo que no venga.
    """
    item = dict(item)
    if not item.get("pregunta") and item.get("question"):
        item["pregunta"] = item["question"]
    if not item.get("id"):
        item["id"] = f"q{numero:03d}"

    familia = str(item.get("familia", "")).strip().lower()
    item["familia"] = {
        "numérica": "numerica",
        "numeric": "numerica",
        "numerical": "numerica",
        "extractive": "extractiva",
        "comparative": "comparativa",
    }.get(familia, familia)

    ticker = item.get("ticker")
    if isinstance(ticker, str):
        item["ticker"] = ticker.strip().upper()

    ejercicio = item.get("fiscal_year")
    if isinstance(ejercicio, str):
        digitos = "".join(c for c in ejercicio if c.isdigit())
        if digitos:
            item["fiscal_year"] = int(digitos)

    herramientas = item.get("herramienta_esperada")
    if isinstance(herramientas, str):
        item["herramienta_esperada"] = [
            h.strip() for h in herramientas.replace(",", "|").split("|") if h.strip()
        ]
    return item


def avisos_de(items: list[dict]) -> list[str]:
    avisos = []
    for item in items:
        pid = item["id"]
        familia = item.get("familia")
        if familia not in {"extractiva", "numerica", "comparativa"}:
            avisos.append(f"{pid}: familia '{familia}' no es extractiva, numerica ni comparativa")
        if not item.get("herramienta_esperada"):
            avisos.append(f"{pid}: sin herramienta_esperada; la trayectoria saldrá como n/a")
        if familia in {"numerica", "comparativa"} and item.get("cifra_esperada") is None:
            avisos.append(f"{pid}: familia {familia} sin cifra_esperada")
        if familia in {"extractiva", "comparativa"} and not item.get("ancla_texto"):
            avisos.append(f"{pid}: familia {familia} sin ancla_texto; la cita fundamentada no se puede comprobar")
    return avisos


ruta = buscar_holdout()
crudos = interfaz.cargar_golden(ruta)
if not crudos:
    raise ValueError(f"{ruta} no tiene ninguna pregunta.")
items = [normalizar(item, i) for i, item in enumerate(crudos, 1)]
sin_pregunta = [it["id"] for it in items if not it.get("pregunta")]
if sin_pregunta:
    raise ValueError(f"Estas líneas no traen pregunta: {sin_pregunta}")

config.DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
with RUTA_RECIBIDO.open("w", encoding="utf-8") as f:
    for item in items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
print(f"Fichero: {ruta}")
if "simulado" in ruta.name.lower():
    print(
        "AVISO: esto es el ensayo del notebook 07, no las preguntas del profesor.\n"
        "Mañana deja RUTA_HOLDOUT = None y suelta el fichero nuevo."
    )
print(f"{len(items)} preguntas · " + ", ".join(
    f"{familia} {sum(it.get('familia') == familia for it in items)}"
    for familia in ("extractiva", "numerica", "comparativa")
))
avisos = avisos_de(items)
if avisos:
    print("\nEl fichero se puede ejecutar. Estos campos, si faltan, dejan una métrica en n/a:")
    for aviso in avisos:
        print(f"  - {aviso}")
else:
    print("El fichero trae los campos que puntúan cita, cifra y trayectoria.")

vista_preguntas = pd.DataFrame(items)
columnas_preguntas = [
    c for c in ("id", "familia", "ticker", "fiscal_year", "pregunta")
    if c in vista_preguntas.columns
]
display(vista_preguntas[columnas_preguntas])

Fichero: C:\Users\jdmar\Desktop\Taller NLP\golden\holdout_simulado.jsonl
AVISO: esto es el ensayo del notebook 07, no las preguntas del profesor.
Mañana deja RUTA_HOLDOUT = None y suelta el fichero nuevo.
10 preguntas · extractiva 4, numerica 3, comparativa 3
El fichero trae los campos que puntúan cita, cifra y trayectoria.


,id,familia,ticker,fiscal_year,pregunta
0,ho-001,extractiva,MSFT,2024,¿Cuáles son las principales exposiciones a divisa que declara Microsoft en su 10-K d...
1,ho-002,extractiva,NVDA,2025,¿Cómo describe NVIDIA su exposición directa a las fluctuaciones de los tipos de camb...
2,ho-003,extractiva,AMZN,2025,¿Qué señala Amazon en los riesgos de su 10-K de FY2025 sobre la evolución de la regu...
3,ho-004,extractiva,META,2024,¿A qué atribuye Meta el descenso de su tipo impositivo efectivo en el ejercicio 2024?
4,ho-005,numerica,AAPL,2024,¿Cuánto efectivo generó Apple con sus actividades de explotación en el ejercicio fis...
5,ho-006,numerica,GOOGL,2025,¿Cuál fue el coste de los ingresos de Alphabet en 2025?
6,ho-007,numerica,MSFT,2025,¿Cuál fue el beneficio por acción diluido de Microsoft en el ejercicio fiscal 2025?
7,ho-008,comparativa,MSFT,2025,¿Cuánto creció el gasto en I+D de Microsoft entre los ejercicios fiscales 2024 y 202...
8,ho-009,comparativa,AAPL,2025,"¿Cómo evolucionó el gasto en I+D de Apple entre 2024 y 2025, y qué razones da la com..."
9,ho-010,comparativa,AMZN,2025,"¿Cuánto creció el resultado de explotación de Amazon entre 2024 y 2025, y a qué atri..."


## Ejecución

Perfil `final`, que es el sistema que se defiende. Cada pregunta estrena hilo. Si una falla, se anota y se sigue. Con la latencia ya medida (~23 s) las diez caben en los veinte minutos.

In [4]:
meta_previa = {}
if RUTA_META.is_file():
    meta_previa = json.loads(RUTA_META.read_text(encoding="utf-8"))

ya_evaluado = (
    not REEJECUTAR
    and meta_previa.get("huella") == huella
    and DESTINO.with_suffix(".csv").is_file()
)

if ya_evaluado:
    tabla = pd.read_csv(DESTINO.with_suffix(".csv"))
    segundos = meta_previa.get("segundos_pared")
    print(
        "Este fichero ya está evaluado. No vuelvo a llamar al modelo.\n"
        "Pon REEJECUTAR = True en la primera celda si quieres repetirlo."
    )
else:
    if not config.hay_modelo():
        raise RuntimeError(config.motivo_sin_modelo())
    print(config.resumen_configuracion())
    print()
    comienzo = time.perf_counter()
    tabla = interfaz.evaluar(
        RUTA_RECIBIDO,
        perfil="final",
        guardar_en=DESTINO,
    )
    segundos = time.perf_counter() - comienzo
    meta = {
        "huella": huella,
        "ruta_origen": str(ruta),
        "segundos_pared": segundos,
        "n": len(tabla),
        "modelo": config.modelo_por_defecto(),
    }
    RUTA_META.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nTiempo de pared: {segundos / 60:.1f} min")

resumen = interfaz.resumir(tabla, "holdout")
print()
print(interfaz.formatear_resumen(resumen))

{'modelo': 'gpt-5-mini', 'temperatura': 1, 'modelo_embeddings': 'BAAI/bge-small-en-v1.5', 'k': 5, 'tolerancia_cifra': 0.01, 'limite_llamadas_herramienta': 8, 'limite_llamadas_modelo': 10, 'precio_entrada_usd_por_millon': 0.25, 'precio_salida_usd_por_millon': 2.0, 'fecha_precios': '2026-09-21'}

Evaluando 10 preguntas · perfil 'final' · modelo gpt-5-mini


  [ 1/10] ho-001     acierto=OK cita=OK cifra=·  tray=OK   39.7s  $0.0057  1 llamadas
  [ 2/10] ho-002     acierto=OK cita=OK cifra=·  tray=OK   16.9s  $0.0056  2 llamadas
  [ 3/10] ho-003     acierto=OK cita=OK cifra=·  tray=OK   22.1s  $0.0073  2 llamadas
  [ 4/10] ho-004     acierto=OK cita=OK cifra=·  tray=OK   10.3s  $0.0035  1 llamadas
  [ 5/10] ho-005     acierto=OK cita=·  cifra=OK tray=OK    6.2s  $0.0042  2 llamadas
  [ 6/10] ho-006     acierto=OK cita=·  cifra=OK tray=OK    5.8s  $0.0039  2 llamadas
  [ 7/10] ho-007     acierto=OK cita=·  cifra=OK tray=OK    7.1s  $0.0029  1 llamadas
  [ 8/10] ho-008     acierto=OK cita=OK cifra=OK tray=OK   19.4s  $0.0068  3 llamadas
  [ 9/10] ho-009     acierto=OK cita=OK cifra=OK tray=OK   19.2s  $0.0051  3 llamadas
  [10/10] ho-010     acierto=no cita=OK cifra=OK tray=OK   18.9s  $0.0078  3 llamadas

Guardado en C:\Users\jdmar\Desktop\Taller NLP\resultados\holdout_clase.csv y .jsonl

  acierto  90.0% · cita 100.0% · cifra 100.0% · trayec

## Lo que se dice en la presentación

La comparación que pide el enunciado es el delta contra el golden propio, no contra el 100 %. Con diez preguntas, una sola mueve diez puntos: por debajo de eso no se interpreta.

In [5]:
def porcentaje(valor) -> str:
    if valor is None or pd.isna(valor):
        return "n/a"
    return f"{100 * float(valor):.1f} %"


def puntos(valor) -> float | None:
    if valor is None or pd.isna(valor):
        return None
    return 100 * float(valor)


def marca(valor) -> str:
    if valor is None or pd.isna(valor):
        return "·"
    if isinstance(valor, str):
        return valor
    return "OK" if bool(valor) else "no"


comparativa = pd.read_csv(config.DIR_RESULTADOS / "tabla_comparativa.csv")
propio = comparativa[
    (comparativa.conjunto == "propio") & (comparativa.sistema == "final")
].iloc[0]
oficial = comparativa[
    (comparativa.conjunto == "oficial") & (comparativa.sistema == "final")
].iloc[0]

filas = []
for etiqueta, origen, claves in (
    ("acierto", resumen.get("acierto"), "acierto"),
    ("extractiva", resumen.get("acierto_extractiva"), "extractiva"),
    ("numérica", resumen.get("acierto_numerica"), "numérica"),
    ("comparativa", resumen.get("acierto_comparativa"), "comparativa"),
    ("cita", resumen.get("cita"), "cita"),
    ("cifra", resumen.get("cifra"), "cifra"),
    ("trayectoria", resumen.get("trayectoria"), "trayectoria"),
):
    filas.append({
        "": etiqueta,
        "golden propio": porcentaje(propio[claves]),
        "holdout": porcentaje(origen),
        "golden oficial": porcentaje(oficial[claves]),
    })
filas.append({
    "": "coste medio",
    "golden propio": f"${float(propio['coste medio ($)']):.4f}",
    "holdout": f"${resumen.get('coste_medio_usd', 0):.4f}",
    "golden oficial": f"${float(oficial['coste medio ($)']):.4f}",
})
filas.append({
    "": "latencia media",
    "golden propio": f"{float(propio['latencia (s)']):.0f} s",
    "holdout": f"{resumen.get('latencia_media_s', 0):.0f} s",
    "golden oficial": f"{float(oficial['latencia (s)']):.0f} s",
})
tabla_delta = pd.DataFrame(filas).set_index("")
display(tabla_delta)

acierto_h = puntos(resumen.get("acierto"))
acierto_p = puntos(propio["acierto"])
delta = None if acierto_h is None else acierto_h - acierto_p
por_pregunta = 100 / max(len(tabla), 1)

puntuables = tabla["acierto"].notna()
n_aciertos = int(tabla.loc[puntuables, "acierto"].astype(bool).sum()) if puntuables.any() else 0
n_errores = int(tabla["error"].notna().sum()) if "error" in tabla else 0

if delta is None:
    frase_delta = "No hay acierto calculable: revisa la columna de error."
elif abs(delta) < por_pregunta:
    frase_delta = (
        f"La diferencia ({delta:+.0f} puntos) es menor que una pregunta "
        f"({por_pregunta:.0f} puntos). No se distingue del ruido: en las ciegas "
        "el sistema se comporta como en el conjunto con el que se iteró."
    )
elif delta < 0:
    frase_delta = (
        f"Baja {abs(delta):.0f} puntos, más de una pregunta. Parte de lo medido "
        "en el golden propio era ajuste a ese conjunto, no solo el procedimiento. "
        "Es el hallazgo que pide la defensa."
    )
else:
    frase_delta = (
        f"Sube {delta:.0f} puntos, más de una pregunta. El procedimiento aguanta "
        "fuera del conjunto con el que se iteró."
    )

if segundos is None:
    frase_tiempo = "El tiempo de pared no quedó registrado en esta ejecución."
elif segundos <= 20 * 60:
    frase_tiempo = f"El proceso entero ha tardado {segundos / 60:.1f} min, dentro de los veinte."
else:
    frase_tiempo = f"El proceso entero ha tardado {segundos / 60:.1f} min, por encima de los veinte."

por_id = {item["id"]: item for item in items}


def porque(fila) -> str:
    if pd.notna(fila.get("error")):
        return f"excepción: {fila['error']}"
    trozos = []
    if marca(fila.get("cifra")) == "no":
        trozos.append("la cifra no cuadra con XBRL")
    if marca(fila.get("cita")) == "no":
        trozos.append("la cita no está en el fragmento")
    if marca(fila.get("fundamentada")) == "no":
        trozos.append("la cita no es la frase que responde")
    if marca(fila.get("trayectoria")) == "no":
        trozos.append("no pasó por la herramienta que tocaba")
    if not trozos:
        trozos.append("el acierto de familia no se cumple")
    esperada = (por_id.get(fila["id"]) or {}).get("herramienta_esperada")
    usadas = fila.get("herramientas")
    if usadas is None or pd.isna(usadas) or str(usadas).strip() == "":
        usadas = "ninguna"
    return "; ".join(trozos) + f". Usó {usadas}; se esperaba {esperada}."


fallos = tabla[tabla["acierto"] != True]  # noqa: E712
lineas_fallo = []
if fallos.empty:
    lineas_fallo.append("No hay preguntas fallidas.")
else:
    for _, fila in fallos.iterrows():
        lineas_fallo.append(
            f"- {fila['id']} ({fila.get('familia')}, {fila.get('ticker')} FY{fila.get('fiscal_year')}): {porque(fila)}"
        )
        lineas_fallo.append(f"  Pregunta: {fila.get('pregunta')}")
        lineas_fallo.append(f"  Esperado: {fila.get('respuesta_esperada')}")
        lineas_fallo.append(f"  Respondió: {fila.get('respuesta')}")

texto = "\n".join([
    (
        f"En las {len(tabla)} preguntas ciegas el sistema final acierta "
        f"{porcentaje(resumen.get('acierto'))} ({n_aciertos} de {int(puntuables.sum())} puntuables"
        + (f", {n_errores} con error" if n_errores else "")
        + "). "
        f"En el golden propio, el mismo sistema acierta {porcentaje(propio['acierto'])}. "
        f"El delta es de {delta:+.0f} puntos."
        if delta is not None
        else "No se ha podido calcular el delta."
    ),
    "",
    frase_delta,
    "",
    (
        f"Coste medio ${resumen.get('coste_medio_usd', 0):.4f} por pregunta "
        f"(en el propio eran ${float(propio['coste medio ($)']):.4f}), "
        f"latencia media {resumen.get('latencia_media_s', 0):.0f} s, "
        f"{resumen.get('llamadas_medias', 0):.1f} llamadas por pregunta. "
        f"{frase_tiempo} "
        f"El guardrail de cifras saltó {int(resumen.get('guardrail_saltó') or 0)} veces."
    ),
    "",
    "El enrutado no cambia con el conjunto: una cifra sale de get_xbrl_fact, "
    "una explicación de search_filings, y una comparativa son dos consultas XBRL "
    "más el Item 7. Si el dato no está, fuente=ninguna y la cifra vacía.",
    "",
    "Fallos:",
    *lineas_fallo,
])

RUTA_DEFENSA.write_text(texto, encoding="utf-8")
print(f"Guardado en {RUTA_DEFENSA}\n")
display(Markdown(texto.replace("\n", "  \n")))

,golden propio,holdout,golden oficial
,,,
acierto,70.0 %,90.0 %,80.0 %
extractiva,71.4 %,100.0 %,100.0 %
numérica,100.0 %,100.0 %,100.0 %
comparativa,33.3 %,66.7 %,42.9 %
cita,92.3 %,100.0 %,100.0 %
cifra,100.0 %,100.0 %,100.0 %
trayectoria,100.0 %,100.0 %,100.0 %
coste medio,$0.0078,$0.0053,$0.0072
latencia media,23 s,17 s,23 s


Guardado en C:\Users\jdmar\Desktop\Taller NLP\resultados\holdout_clase_defensa.txt



En las 10 preguntas ciegas el sistema final acierta 90.0 % (9 de 10 puntuables). En el golden propio, el mismo sistema acierta 70.0 %. El delta es de +20 puntos.  
  
Sube 20 puntos, más de una pregunta. El procedimiento aguanta fuera del conjunto con el que se iteró.  
  
Coste medio $0.0053 por pregunta (en el propio eran $0.0078), latencia media 17 s, 2.0 llamadas por pregunta. El proceso entero ha tardado 2.8 min, dentro de los veinte. El guardrail de cifras saltó 0 veces.  
  
El enrutado no cambia con el conjunto: una cifra sale de get_xbrl_fact, una explicación de search_filings, y una comparativa son dos consultas XBRL más el Item 7. Si el dato no está, fuente=ninguna y la cifra vacía.  
  
Fallos:  
- ho-010 (comparativa, AMZN FY2025): la cita no es la frase que responde. Usó get_xbrl_fact|get_xbrl_fact|search_filings; se esperaba ['get_xbrl_fact', 'search_filings'].  
  Pregunta: ¿Cuánto creció el resultado de explotación de Amazon entre 2024 y 2025, y a qué atribuye la dirección la mejora de AWS?  
  Esperado: El resultado consolidado creció, y la mejora de AWS se atribuye al aumento de las ventas, compensado en parte por el gasto en infraestructura tecnológica para sostener ese crecimiento. [FY2024: 68,593,000,000 → FY2025: 79,975,000,000 USD]  
  Respondió: El resultado de explotación pasó de $68,593,000,000 en 2024 a $79,975,000,000 en 2025, un aumento absoluto de $11,382,000,000 (≈16.6%). La dirección atribuye la mejora de AWS principalmente al aumento del uso por parte de los clientes, parcialmente compensado por cambios de precios vinculados a contratos a largo plazo.

In [6]:
vista = tabla.copy()
for columna in ("acierto", "cita", "cifra", "trayectoria", "fundamentada"):
    if columna in vista:
        vista[columna] = vista[columna].map(marca)
columnas = [
    c for c in (
        "id", "familia", "acierto", "cita", "cifra", "trayectoria",
        "fundamentada", "herramientas", "latencia_s", "coste_usd", "respuesta",
    )
    if c in vista.columns
]
display(vista[columnas])

,id,familia,acierto,cita,cifra,trayectoria,fundamentada,herramientas,latencia_s,coste_usd,respuesta
0,ho-001,extractiva,OK,OK,·,OK,OK,search_filings,39.685661,0.005692,Las principales exposiciones a divisa que declara Microsoft en su 10‑K FY2024 son: e...
1,ho-002,extractiva,OK,OK,·,OK,OK,list_available|search_filings,16.947103,0.005644,NVIDIA declara que su exposición directa a las fluctuaciones de los tipos de cambio ...
2,ho-003,extractiva,OK,OK,·,OK,OK,list_available|search_filings,22.142707,0.007287,Amazon indica que el marco regulatorio que le afecta está evolucionando y expandiénd...
3,ho-004,extractiva,OK,OK,·,OK,OK,search_filings,10.337076,0.003517,Meta atribuye la disminución de su tipo impositivo efectivo en 2024 principalmente a...
4,ho-005,numerica,OK,·,OK,OK,·,list_available|get_xbrl_fact,6.176808,0.004217,"Apple generó 118,254,000,000 USD de efectivo procedente de sus actividades de explot..."
5,ho-006,numerica,OK,·,OK,OK,·,list_available|get_xbrl_fact,5.827640,0.003942,El coste de los ingresos (Cost of Revenue) de Alphabet en el ejercicio fiscal 2025 f...
6,ho-007,numerica,OK,·,OK,OK,·,get_xbrl_fact,7.148403,0.002886,El beneficio por acción diluido de Microsoft en el ejercicio fiscal 2025 fue de 13.6...
7,ho-008,comparativa,OK,OK,OK,OK,OK,get_xbrl_fact|get_xbrl_fact|search_filings,19.423340,0.006808,"El gasto en I+D aumentó de $29,510,000,000 en el ejercicio fiscal 2024 a $32,488,000..."
8,ho-009,comparativa,OK,OK,OK,OK,OK,get_xbrl_fact|get_xbrl_fact|search_filings,19.248552,0.005081,"El gasto en I+D de Apple aumentó de 31,370,000,000 USD en el ejercicio 2024 a 34,550..."
9,ho-010,comparativa,no,OK,OK,OK,no,get_xbrl_fact|get_xbrl_fact|search_filings,18.871246,0.007826,"El resultado de explotación pasó de $68,593,000,000 en 2024 a $79,975,000,000 en 202..."
